# Entrenamiento global + local controlado por YAML

Este es el notebook principal para entrenar. No reconstruye manualmente el pipeline: usa el mismo high level que la terminal. El flujo es **elegir YAML → validar → opcionalmente sobrescribir para una prueba → entrenar**.

Configuraciones listas:

- `configs/training/default_train.yaml`: entrenamiento original; no descarga FG-NET/AgeDB.
- `configs/training/paired_fgnet_train.yaml`: activa FG-NET longitudinal. Recomendado para la primera prueba.
- `configs/training/paired_agedb_train.yaml`: activa AgeDB longitudinal. Es más grande y heterogéneo.

In [ ]:
# 1. Ubicar el repositorio e importar el único high level de entrenamiento.
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.common import print_config_summary
from scripts.train_cli import load_training_config, run_training

print(f"Repositorio: {REPO_ROOT}")

## 2. Elegir y validar la corrida

Cambie solamente `CONFIG_PATH`. Los YAML `paired_*_train.yaml` heredan todos los valores recomendados de `default_train.yaml` y añaden la fuente longitudinal. Los parámetros propios de FG-NET/AgeDB viven en `configs/data/paired_*.yaml`.

Esta celda **no carga modelos, no crea dataloaders y no descarga datos**. Solo combina y muestra la configuración efectiva.

In [ ]:
CONFIG_PATH = "configs/training/paired_fgnet_train.yaml"
# CONFIG_PATH = "configs/training/default_train.yaml"
# CONFIG_PATH = "configs/training/paired_agedb_train.yaml"

config = load_training_config(CONFIG_PATH)
print_config_summary(config)

paired = config["paired_supervision"]
print("Config validado:", CONFIG_PATH)
print("Supervisión longitudinal:", paired["enabled"])
if paired["enabled"]:
    print("Dataset:", paired["dataset"])
    print("Caché persistente:", paired["cache_dir"])
    print("Frecuencia/peso:", paired["every_n_steps"], paired["weight"])

## 3. Sobrescrituras temporales (opcional)

Para una corrida real, deje `QUICK_TEST = False`: se usarán exactamente los hiperparámetros del YAML. Para verificar que todo conecta en su máquina, active `QUICK_TEST`; limita batches y épocas solo en memoria y no modifica ningún archivo.

Los controles principales son `training`, `losses`, `adapters`, `paired_supervision` y `sampling`. Para experimentos reproducibles, haga el cambio definitivo en el YAML y cambie también `run.name`.

In [ ]:
QUICK_TEST = False

if QUICK_TEST:
    config["run"]["name"] += "_smoke"
    config["training"]["local_num_epochs"] = 1
    config["training"]["global_num_epochs"] = 1
    config["training"]["local_max_batches"] = 2
    config["training"]["global_max_batches"] = 2

print("Run:", config["run"]["name"])
print("Épocas local/global:", config["training"]["local_num_epochs"], config["training"]["global_num_epochs"])
print("Límite batches local/global:", config["training"]["local_max_batches"], config["training"]["global_max_batches"])

## 4. Entrenar

Cambie `START_TRAINING` a `True` cuando haya revisado el resumen. En ese momento se cargan datos/modelos. Si la supervisión longitudinal está activa, se revisa primero la caché configurada:

- caché completa: se reutiliza inmediatamente;
- ZIP descargado pero no extraído: se reutiliza y extrae;
- nada disponible: se descarga una sola vez desde Kaggle y se verifica.

Con `default_train.yaml` no se consulta ni descarga ningún dataset adicional.

In [ ]:
START_TRAINING = False

if not START_TRAINING:
    print("Entrenamiento detenido de forma segura. Revise la configuración y cambie START_TRAINING=True.")
else:
    result = run_training(config)
    print("Resultado:")
    print(result)

## Referencias rápidas

- Guía completa: `docs/README.md`.
- Ancla de hiperparámetros: `configs/training/default_train.yaml`.
- Entrada high level: `scripts/train_cli.py`.
- Descarga, pares y split longitudinal: `data/paired_aging_dataset.py`.
- Loss longitudinal: `src/loss/paired_supervision_loss.py`.
- Loop que intercala la loss: `src/training/train_aging_model.py`.